In [1]:
import pandas as pd

# Getting the Scoring Sequence for each divison

In [2]:
nfl_divisions = {
    "AFC-East":  ("BUF", "MIA", "NE", "NYJ"),
    "AFC-North": ("BAL", "CIN", "CLE", "PIT"),
    "AFC-South": ("HOU", "IND", "JAX", "TEN"),
    "AFC-West":  ("DEN", "KC", "LV", "LAC"),

    "NFC-East":  ("DAL", "NYG", "PHI", "WAS"),
    "NFC-North": ("CHI", "DET", "GB", "MIN"),
    "NFC-South": ("ATL", "CAR", "NO", "TB"),
    "NFC-West":  ("ARI", "LAR", "SF", "SEA")
}

rows = []

for season in range(2002, 2025):
    for division, teams in nfl_divisions.items():

        teams = list(teams)

        # Las Vegas Raiders were Oakland Raiders through 2019
        if season <= 2019 and "LV" in teams:
            teams[teams.index("LV")] = "OAK"

        # Los Angeles Chargers were San Diego Chargers through 2016
        if season <= 2016 and "LAC" in teams:
            teams[teams.index("LAC")] = "SD"

        # Los Angeles Rams were St. Louis Rams through 2015
        if season <= 2015 and "LAR" in teams:
            teams[teams.index("LAR")] = "STL"

        rows.append({
            "season": season,
            "league": "NFL",
            "division": division,
            "division_teams": tuple(teams)
        })

division_df = pd.DataFrame(rows)

games_df = pd.read_csv("../Data/game_data_us_leagues.csv")

# Make one row per season/team/division
team_division_df = (
    division_df
    .explode("division_teams")
    .rename(columns={"division_teams": "team"})
)

# this is only the NFL games
nfl_games = games_df[
    (games_df["league"] == "NFL") &
    (games_df["season"].between(2002, 2024))
].copy()

# adding team 1 division
nfl_games = nfl_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team1"],
    right_on=["season", "team"],
    how="left"
)

nfl_games = (
    nfl_games
    .rename(columns={"division": "division1"})
    .drop(columns="team")
)

# adds team 2 division
nfl_games = nfl_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team2"],
    right_on=["season", "team"],
    how="left"
)

nfl_games = (
    nfl_games
    .rename(columns={"division": "division2"})
    .drop(columns="team")
)

# FILTERING BY DIVISION
div_games = nfl_games[
    nfl_games["division1"] == nfl_games["division2"]
].copy()

div_games["division"] = div_games["division1"]

div_games = div_games.drop(columns=["division1", "division2"])

# go game by game and figure out the winner
def get_winner(row):
    if row["score1"] > row["score2"]:
        return row["team1"]
    elif row["score2"] > row["score1"]:
        return row["team2"]
    else:
        return None


div_games["winner"] = div_games.apply(
    get_winner,
    axis=1
)

# pairs need to have same ordering so when we group we dont double count
teams = div_games[["team1", "team2"]]

div_games["team1"] = teams.min(axis=1)
div_games["team2"] = teams.max(axis=1)

# mark whoever won a division game
div_games["win1"] = (
    div_games["winner"] == div_games["team1"]
).astype(int)

div_games["win2"] = (
    div_games["winner"] == div_games["team2"]
).astype(int)

# compute the season series
series_df = (
    div_games
    .groupby(
        ["season", "division", "team1", "team2"],
        as_index=False
    )
    .agg(
        wins1=("win1", "sum"),
        wins2=("win2", "sum")
    )
)

# determine the full series winner
def get_series_result(row):
    if row["wins1"] > row["wins2"]:
        return 1
    elif row["wins2"] > row["wins1"]:
        return -1
    else:
        return 0


series_df["result"] = series_df.apply(
    get_series_result,
    axis=1
)

series_df.insert(
    1,
    "league",
    "NFL"
)

series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2"
    ]
]

# GET THE SERIES WINS FOR EACH TEAM

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    team1["result"] == 1
).astype(int)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    team2["result"] == -1
).astype(int)

# create scoring sequences
score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)

In [ ]:
score_sequences.to_csv('../Analysis/nfl_scoring_sequence.csv', index=False)

# Getting upset frequencies per division

In [4]:
# lookup each team's score for a given season and division
score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# determine whether each series is an upset
def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# count upsets for each season and division
upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )["upset"]
    .sum()
)

In [ ]:
upset_counts.to_csv('../Analysis/nfl_upset_counts.csv', index=False)